In [0]:


with results as (
    select 
        *,
        CASE WHEN `Synthetic True Propensity` > 50 THEN 1 ELSE 0 END AS synthetic_propensity_label,
        NTILE(10) OVER (ORDER BY propensity_score DESC) AS bucket
    from workspace.default.synthetic_propensity_model_scoring_data t1
    left join workspace.default.customer_propensity_predictions t2 on t2.customer_id = t1.`Customer ID`
),

tmp as (
    select 
        bucket,
        count(distinct case when results.synthetic_propensity_label = 1 then customer_id else null end) as num_positive_actual,
        count(distinct customer_id) as total,
        count(distinct case when results.propensity_label = 1 then customer_id else null end) as num_positive_predicted
    from results
    group by bucket
)

-- Lift Analysis
select
    *,
    sum(num_positive_actual) over () as positive_actual_total,
    sum(total) over () as total_,
    positive_actual_total / total_ as baseline,
    num_positive_actual / total as actual_cvr,
    actual_cvr / baseline as lift
from tmp
;
